In [1]:
import math
import time
import socket
import numpy as np

from scipy.signal import resample_poly
from scipy.io.wavfile import write as wavwrite

from pynq import allocate
from pynq.lib import MicroblazeLibrary
from pynq.lib.pmod import Pmod_ADC
from pynq.overlays.base import BaseOverlay

from multiprocessing import Process, Queue

# Global Setup
Here we initialize sockets, the BaseOverlay, the PMODs, etc.

In [36]:
base = BaseOverlay("base.bit")

# set up ADC for thermometer
adc = Pmod_ADC(base.PMODA)

# set up client socket
# HOST = "192.168.2.0"
HOST = "0.0.0.0"
PORT = 12345
client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# Microphone - PMOD B
This section is for sampling the microphone, doing some small signal processing stuff, and then sending the audio over the wire to the server

In [37]:
%%microblaze base.PMODB
#include <i2c.h>
#include <time.h>
#include <yield.h>
#include <pyprintf.h>

#define DIGILENT_SCL 2
#define DIGILENT_SDA 3
#define CH_0 16
#define CH_1 32
#define CH_0_1 48

#define MAX_SAMPLES 4096

size_t read_idx;
size_t write_idx;
static int samples[MAX_SAMPLES];    

i2c device;

int init_adc() {
    device = i2c_open(DIGILENT_SDA, DIGILENT_SCL);
    unsigned char buf[2];
    buf[0] = CH_0;
    i2c_write(device, 0x28, buf, 1);
    return 0;
}

int read_adc() {
    unsigned char buf[2];
    i2c_read(device, 0x28, buf, 2);
    return ((buf[0] & 0x0F) << 8) | buf[1];
}

int read_adc_multi(size_t size) {
    size_t i;
    unsigned char r[2];
    for (i = 0; i < size; ++i) {
        i2c_read(device, 0x28, r, 2);
        
        samples[i] = ((r[0] & 0x0F) << 8) | r[1];
    }
    
    return 0;
}

int read_adc_dma(void *buf, size_t size) {
    size_t i;
    int* int_data = (int*)buf;
    unsigned char r[2];
    for (i = 0; i < size; ++i) {
        i2c_read(device, 0x28, r, 2);
        
        int_data[i] = ((r[0] & 0x0F) << 8) | r[1];
    }
    return 0;
}

// writes samples to the circular buffer
void update_circular() {
    unsigned char r[2];
    while (1) {
        while (((write_idx + 1) % MAX_SAMPLES)
               != (read_idx % MAX_SAMPLES)) {
            i2c_read(device, 0x28, r, 2);
            samples[write_idx] = ((r[0] & 0x0F) << 8) | r[1];
            write_idx = ++write_idx % MAX_SAMPLES;
        }
        yield();
    }
}

int get_circular() {
    int save = 0;
    if (read_idx == write_idx)
    {
        // buffer is empty
        return -1;
    }
    
    save = samples[read_idx++];
    read_idx %= MAX_SAMPLES;
    return save;
}

int get_sample(size_t i) {
    return samples[i];
}

In [38]:
def despike(audio, threshold=3.0):
    mean = np.mean(audio)
    std = np.std(audio)
    spikes = np.abs(audio - mean) > threshold * std
    indices = np.arange(len(audio))
    audio[spikes] = np.interp(indices[spikes], indices[~spikes], audio[~spikes])
    return audio

def normalize(audio):
    peak = np.max(np.abs(audio))
    if peak == 0:
        return audio
    return audio / peak * 32767.0

def sample_microphone(queue: Queue):
    CHUNK = 4096
    processed = []
    shared_buf = allocate(shape=(CHUNK,), dtype=np.int32)
    shared_buf[:] = [0] * CHUNK

    total_samples = 0
    total_time = 0.0
    
    good = "Good noise level ({})"
    bad = "Too loud!! ({})"
    
    while True:
        sample_begin = time.time()
        read_adc_dma(shared_buf, CHUNK)
        sample_end = time.time()

        total_time += sample_end - sample_begin
        total_samples += CHUNK
        
        mv = max(shared_buf)
        
        dBV = 20 * math.log((mv / 4096) * 3.3)
        
        if dBV < 55.0:
            status = good.format(dBV)
        else:
            status = bad.format(dBV)
        
        queue.put(("mic_level", dBV))
        queue.put(("mic_status", status))

        chunk = shared_buf[:CHUNK].astype(np.float32)
        upsampled = resample_poly(chunk, 4, 1).astype(np.int16)
        processed.extend(upsampled)

        sample_rate = int(total_samples / total_time)

        audio = np.array(processed, dtype=np.float32)

        audio -= np.mean(audio)

        audio = despike(audio, threshold=3.0)
        audio = normalize(audio)
        
        # send to server



# Temperature Sensor - PMOD A
This code is for sampling the analog temperature sensor and sending it over the wire to the server board

In [39]:
def sample_thermometer(queue: Queue):
    R1 = 10000.0
    VCC = 2.048

    c1 = 0.001129148
    c2 = 0.000234125
    c3 = 0.0000000876741

    Temp_offset = 17.8  # adjust to known temp reading
    Too_Hot = 75.0      # temp thresholds for alerts
    Too_Cold = 65.0

    while True:
        voltage = adc.read(1, 0, 0)[0] # read channel 1, single-ended, unipolar

        if 0 < voltage < VCC:
            R2 = R1 * ((VCC / voltage) - 1.0)

            logR2 = math.log(R2)
            kelvin = 1.0 / (c1 + c2 * logR2 + c3 * (logR2 ** 3))
            celsius = kelvin - 273.15

            celsius_corrected = celsius + Temp_offset
            fahrenheit_corrected = (celsius_corrected * 9.0) / 5.0 + 32.0

            if celsius_corrected >= Too_Hot:
                status = "WARNING: Too Hot!"
            elif celsius_corrected <= Too_Cold:
                status = "WARNING: Too Cold!"
            else:
                status = "Temperature is comfortable."
            
            queue.put(("temp_c", celsius_corrected))
            queue.put(("temp_f", fahrenheit_corrected))
            queue.put(("temp_status", status))

            print(f"Temperature: {celsius_corrected:.2f}°C | {fahrenheit_corrected:.2f}°F | {status}")
        else:
            print("Invalid reading")

        time.sleep(1)

# Main Loop
Here we dispatch the microphone sampling and the thermometer sampling to separate processes

In [51]:
init_adc()

queue = Queue()

mic_p = Process(target=sample_microphone, args=(queue,))
tmp_p = Process(target=sample_thermometer, args=(queue,))

tmp_p.start()
mic_p.start()


Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperature: 67.58°C | 153.64°F | Temperature is comfortable.
Temperat

Process Process-17:
Process Process-18:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_19756/2368810146.py", line 57, in sample_microphone
    audio = despike(audio, threshold=3.0)
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_19756/2368810146.py", line 5, in despike
    indices = np.arange(len(audio))
KeyboardInterrupt
  File "/tmp/ipykernel_19756/2683562931.py", line 41, in sample_thermometer
    time.sleep(1)
KeyboardInterrupt


In [52]:
payload = {
    "temp_c": 0,
    "temp_f": 0,
    "mic_level": 0,
    "temp_status": "",
    "mic_status": ""
}

while True:
    for i in range(20):
        payload.update([queue.get()])
    client.sendto(
        b",".join([str(x).encode("utf-8") for x in payload.values()]),
        ("192.168.2.99", PORT)
    )
    print("sent")

sent
sent
sent
sent
sent
sent
sent


KeyboardInterrupt: 

In [50]:
mic_p.kill()

tmp_p.kill()

In [47]:
queue.get()

('temp_c', 67.57848540100751)